In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader,TensorDataset
import torch.optim as optim

In [2]:
df = pd.read_csv('DateFruit_Dataset (1).csv')

In [3]:
df

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.9080,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.1440,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.3940,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.2100,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.3540,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
893,255403,1925.3650,691.8453,477.1796,0.7241,570.2536,0.9785,261028,0.7269,1.4499,...,2.2423,2.3704,2.7202,-25296416768,-19168882688,-18473392128,49.0869,43.0422,42.4153,SOGAY
894,365924,2664.8230,855.4633,551.5447,0.7644,682.5752,0.9466,386566,0.6695,1.5510,...,3.4109,3.5805,3.9910,-31605219328,-21945366528,-19277905920,46.8086,39.1046,36.5502,SOGAY
895,254330,1926.7360,747.4943,435.6219,0.8126,569.0545,0.9925,256255,0.7240,1.7159,...,2.2759,2.5090,2.6951,-22242772992,-19594921984,-17592152064,44.1325,40.7986,40.9769,SOGAY
896,238955,1906.2679,716.6485,441.8297,0.7873,551.5859,0.9604,248795,0.6954,1.6220,...,2.6769,2.6874,2.7991,-26048595968,-21299822592,-19809978368,51.2267,45.7162,45.6260,SOGAY


In [4]:
df['Class'].unique()

<ArrowStringArray>
['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY']
Length: 7, dtype: str

In [5]:
df.isnull().sum()

AREA             0
PERIMETER        0
MAJOR_AXIS       0
MINOR_AXIS       0
ECCENTRICITY     0
EQDIASQ          0
SOLIDITY         0
CONVEX_AREA      0
EXTENT           0
ASPECT_RATIO     0
ROUNDNESS        0
COMPACTNESS      0
SHAPEFACTOR_1    0
SHAPEFACTOR_2    0
SHAPEFACTOR_3    0
SHAPEFACTOR_4    0
MeanRR           0
MeanRG           0
MeanRB           0
StdDevRR         0
StdDevRG         0
StdDevRB         0
SkewRR           0
SkewRG           0
SkewRB           0
KurtosisRR       0
KurtosisRG       0
KurtosisRB       0
EntropyRR        0
EntropyRG        0
EntropyRB        0
ALLdaub4RR       0
ALLdaub4RG       0
ALLdaub4RB       0
Class            0
dtype: int64

In [6]:
X = df.drop(['Class'],axis=1)
y= df['Class']

In [7]:
from sklearn.preprocessing import StandardScaler,LabelEncoder
encoder = LabelEncoder()
y = encoder.fit_transform(y)

In [8]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [9]:
scaler = StandardScaler()
X_scaled_train = scaler.fit_transform(X_train)
X_scaled_test = scaler.transform(X_test)

In [10]:
from torch.utils.data import DataLoader,TensorDataset

X_train_tensor = torch.tensor(X_scaled_train,dtype=torch.float32)
y_train_tensor = torch.tensor(y_train,dtype=torch.long)

# In terms of multiclass classfication lables are tends to be long format.

X_test_tensor = torch.tensor(X_scaled_test,dtype=torch.float32)
y_test_tensor = torch.tensor(y_test,dtype=torch.long)

In [11]:
train_dataset = TensorDataset(X_train_tensor,y_train_tensor)
test_dataset = TensorDataset(X_test_tensor,y_test_tensor)

train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=True)

In [12]:
# Building ann model

class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()

        self.model = nn.Sequential(
            nn.Linear(X_train.shape[1],64),
            nn.ReLU(),
            nn.Linear(64,64),
            nn.ReLU(),
            nn.Linear(64,7),
        )
    def forward(self,x):
        return self.model(x)

In [13]:
model = ANN()
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [14]:
# Traininig the model
epoch_avg = []
epoch = 100
for i in range(epoch):
    model.train()

    running_loss = 0.0

    for xb,yb in train_loader:
        optimizer.zero_grad()
        output = model(xb)    
        loss = criteria(output,yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_avg_loss = running_loss/len(train_loader)
    epoch_avg.append(epoch_avg_loss)

    print(f'{i+1}/{epoch} => LOSS {epoch_avg_loss}')

1/100 => LOSS 1.71657323318979
2/100 => LOSS 1.1043517097182896
3/100 => LOSS 0.7140187958012456
4/100 => LOSS 0.5412068457707114
5/100 => LOSS 0.45469450043595355
6/100 => LOSS 0.39714479187260504
7/100 => LOSS 0.3575324526299601
8/100 => LOSS 0.3212336126876914
9/100 => LOSS 0.3104821197364641
10/100 => LOSS 0.2665617938922799
11/100 => LOSS 0.2524171452159467
12/100 => LOSS 0.2303835419209107
13/100 => LOSS 0.22175361313249753
14/100 => LOSS 0.20944400909154312
15/100 => LOSS 0.1983712247532347
16/100 => LOSS 0.19003716629484427
17/100 => LOSS 0.18102520509906436
18/100 => LOSS 0.17232423923585727
19/100 => LOSS 0.17418449955142062
20/100 => LOSS 0.1590551390596058
21/100 => LOSS 0.16106674577230992
22/100 => LOSS 0.15180477694324826
23/100 => LOSS 0.1423609596879586
24/100 => LOSS 0.13708078116178513
25/100 => LOSS 0.13653636529393817
26/100 => LOSS 0.13053485929318095
27/100 => LOSS 0.13294325396418571
28/100 => LOSS 0.1370268549932086
29/100 => LOSS 0.12797451553785283
30/100 => 

In [23]:
# evaluation part

model.eval()
total = 0
correct = 0

for xb,yb in test_loader:
    output = model(xb)

    _ , predicted_index = torch.max(output,1)
    correct += (predicted_index==yb).sum().item()
    total += yb.size(0)

print(f"Total value : {total}")
print(f"correct value : {correct}")

# accuracy:

print(f"Accuracy : {(correct/total)*100}")

Total value : 180
correct value : 170
Accuracy : 94.44444444444444
